In [ ]:
import sys, platform, os
print('executable:', sys.executable)
print('platform:', platform.platform())
print('COLAB_TAG:', os.environ.get('COLAB_RELEASE_TAG', 'NOT SET'))
print('cwd:', os.getcwd())
print('Colab:', os.environ.get('COLAB_RELEASE_TAG') is not None)

In [ ]:
import sys, os
print("=== Colab Infrastructure Verification ===")
print(f"Runtime: {'Colab' if os.environ.get('COLAB_RELEASE_TAG') else 'Local'}")
print(f"COLAB_TAG: {os.environ.get('COLAB_RELEASE_TAG', 'N/A')}")
print(f"Python: {sys.executable}")
print(f"CWD: {os.getcwd()}")

# Verify colab_check.py logic works
in_colab = bool(os.environ.get("COLAB_RELEASE_TAG"))
print(f"\nis_colab() = {in_colab}")

# Check GPU availability
try:
    import torch
    gpu = torch.cuda.is_available()
    print(f"GPU available: {gpu}")
    if gpu:
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("torch not installed — GPU check skipped")

print("\n=== Colab kernel confirmed ready for heavy work ===")
print("NOTE: Push code to GitHub first, then run colab_run_all.py")

In [ ]:
import os, sys
print(sys.executable)
print(os.getcwd())
print(os.environ.get("COLAB_RELEASE_TAG"))

In [ ]:
!git clone https://github.com/dragonscypher/Marketify.git /content/Marketify 2>/dev/null || (cd /content/Marketify && git pull)
!python /content/Marketify/scripts/colab_bootstrap.py

In [ ]:
# Fresh clone + full pipeline test
!cd /content && rm -rf Marketify && git clone https://github.com/dragonscypher/Marketify.git
%cd /content/Marketify
!python -m pip install -q -e .
!python -c "import marketify, marketify.data.market_data; print('imports ok')"
!python scripts/audit_repo_files.py
!python scripts/colab_check.py
!MARKETIFY_REQUIRE_COLAB=1 python scripts/colab_run_all.py

In [ ]:
!cd /content && rm -rf Marketify && git clone https://github.com/dragonscypher/Marketify.git
%cd /content/Marketify
!git log --oneline -3
!python -m pip install -e . -q
!python -c "import marketify, marketify.data.market_data; print('imports ok')"
!python scripts/colab_check.py
!MARKETIFY_REQUIRE_COLAB=1 python scripts/colab_run_all.py

In [ ]:
import subprocess
r = subprocess.run(["cat", "marketify/backtest/benchmark.py"], capture_output=True, text=True)
print(r.stdout[:1500])

In [ ]:
!cd /content && rm -rf Marketify && git clone https://github.com/dragonscypher/Marketify.git

%cd /content/Marketify

!git log --oneline -3
!ls -la
!ls -la scripts

!python scripts/colab_check.py


!python -m pip install -r requirements-colab.txt 
!python -m pip install -e . 

!MARKETIFY_REQUIRE_COLAB=1 python scripts/colab_run_all.py

!python scripts/run_exit_rule_experiment.py
!python scripts/run_multimodal_benchmark.py
!ls -la reports

!cat reports/real_benchmark.md
!cat reports/strategy_failure_deep_dive.md
!cat reports/tuning_leaderboard.csv
!cat reports/exit_rule_experiment.md
!cat reports/exit_rule_experiment.csv

!cat reports/NEXT_STATUS.


In [ ]:
# === Validation + Automation + Same-Path Real Benchmark (Colab only) ===
%cd /content
!rm -rf Marketify
!git clone https://github.com/dragonscypher/Marketify.git
%cd /content/Marketify
!git log --oneline -3
!python scripts/colab_check.py
!python -m pip install -r requirements-colab.txt -q
!python -m pip install -e . -q
!python -m pytest tests/ -q
!python scripts/validate_marketify.py
!python scripts/train_and_check.py
!python scripts/compare_models.py
!cat reports/validation_summary.md
!cat reports/broker_validation.md
!cat reports/model_leaderboard.md
!cat reports/daily_stress_iteration_log.md
!cat reports/NEXT_STATUS.md

In [ ]:
# === Fresh clone summary only ===
%cd /content/Marketify
import json, re, subprocess
from pathlib import Path
real_benchmark = Path('reports/real_benchmark.md').read_text(encoding='utf-8') if Path('reports/real_benchmark.md').exists() else ''
next_status = Path('reports/NEXT_STATUS.md').read_text(encoding='utf-8') if Path('reports/NEXT_STATUS.md').exists() else ''
validation = json.loads(Path('reports/validation_summary.json').read_text(encoding='utf-8')) if Path('reports/validation_summary.json').exists() else {}
training = json.loads(Path('reports/training_check_summary.json').read_text(encoding='utf-8')) if Path('reports/training_check_summary.json').exists() else {}
def pick(key: str) -> str:
    patterns = [rf'^- {re.escape(key)}: (.+)$', rf'^{re.escape(key)}: (.+)$']
    for text in (next_status, real_benchmark):
        for pattern in patterns:
            match = re.search(pattern, text, flags=re.MULTILINE)
            if match:
                return match.group(1).strip()
    return 'MISSING'
def check(name: str) -> dict:
    for item in validation.get('results', []):
        if item.get('name') == name:
            return item
    return {}
print(f'commit hash: {subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()}')
print(f'pytest_result: {"PASS" if check("pytest").get("passed") else "FAIL"}')
print(f'validation_result: {"PASS" if validation.get("all_passed") else "FAIL"}')
print(f'training_check_result: {training.get("training_check_result", "MISSING")}')
print(f'ui_flow_result: {"PASS" if check("scripted_ui_flow_proof").get("passed") else "FAIL"}')
print(f'ui_approval_works: {check("scripted_ui_flow_proof").get("checks", {}).get("approve_paper_fill", False)}')
print(f'restart_reload_works: {check("scripted_ui_flow_proof").get("checks", {}).get("restart_sqlite_reload", False)}')
print(f'broker_readiness_result: {"PASS" if check("broker_readiness").get("passed") else "FAIL"}')
print(f'broker_readiness_details: {check("broker_readiness").get("results", {})}')
print(f'champion_model: {pick("champion_model")}')
print(f'weekly_return_pct: {pick("weekly_return_pct")}')
print(f'daily_return_pct: {pick("daily_return_pct")}')
print(f'sharpe: {pick("sharpe")}')
print(f'max_drawdown_pct: {pick("max_drawdown_pct")}')
print(f'cvar_95_pct: {pick("cvar_95_pct")}')
print(f'trade_count: {pick("trade_count")}')
print(f'approval_count: {pick("approval_count")}')
print(f'keep_coverage_pct: {pick("keep_coverage_pct")}')
print(f'expectancy: {pick("expectancy")}')
print(f'disagreement_reject_rate: {pick("disagreement_reject_rate")}')
print(f'keep_rate_by_gate: {pick("keep_rate_by_gate")}')
print(f'dominant_gate: {pick("dominant_gate")}')
print(f'weekly_benchmark: {pick("weekly_benchmark")}')
print(f'daily_stress_benchmark: {pick("daily_stress_benchmark")}')
print(f'PASS/FAIL: {pick("PASS/FAIL")}')
print(f'exact blocker: {pick("exact_blocker")}')
print(f'daily stress blocker: {pick("daily_stress_exact_blocker")}')